In [ ]:
import torch
import numpy as np
from single_cellm.jointemb.single_cellm_lightning import TranscriptomeTextDualEncoderLightning
from single_cellm.jointemb.processing import TranscriptomeTextDualEncoderProcessor
from single_cellm.config import get_path, model_path_from_name

from single_cellm.jointemb.dataset.jointemb import JointEmbedDataModule


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pl_model = TranscriptomeTextDualEncoderLightning.load_from_checkpoint(snakemake.input.model)
pl_model.eval().to(device)
pl_model.model.prepare_models(
    pl_model.model.transcriptome_model, pl_model.model.text_model, force_freeze=True
)
pl_model.freeze()

In [ ]:
from tqdm.notebook import tqdm

# traverse all transcriptomes, log features and embeddings and save them

dm = JointEmbedDataModule(dataset_name=snakemake.wildcards["dataset"],
                     transcriptome_processor=pl_model.model.transcriptome_model.config.model_type,
                     tokenizer=pl_model.model.text_model.config.model_type,
                     batch_size=32, 
                     train_fraction=0.0)  # use val dataloader to prevent shuffling of datapoints
dm.prepare_data()
dm.setup()
dl = dm.val_dataloader()
results = []

for batch in tqdm(dm.val_dataloader(), desc=f"Processing {snakemake.wildcards['dataset']}"):
    result = pl_model(**{k: t.to(device) for k, t in batch.items()})
    result["transcriptome_text_similarity"] = torch.diag(result["logits_per_transcriptome"])
    results.append({k: t.detach().cpu() for k, t in result.items() if k not in ["logits_per_transcriptome", "logits_per_text"]})

In [ ]:
aggregated_dict = {key: torch.cat([d[key] for d in results]) for key in results[0]}
aggregated_dict["orig_ids"] = dl.dataset.orig_ids
assert len(aggregated_dict["orig_ids"]) == len(aggregated_dict["discrimination"])

In [ ]:
np.savez(snakemake.output["model_outputs"], **aggregated_dict) 